# ML Laboratuvarı · Modül 2, Ünite 9: Bir öngörüyü dürüstçe değerlendirmek

**Prof. Dr. Mehmet Songur** · ML Laboratuvarı, Modül 2

Veri: ABD işsizlik oranı (FRED: `UNRATE`, aylık, %). Tablo: `issizlik` (sütunlar: `tarih`, `oran`). Kodu **Gemini'ye yazdır**, sonra **denetle**.

**Kurallar**
1. **Kurulum** ve **Kontrol** hücrelerini değiştirme; yalnızca çalıştır.
2. Görev hücrelerinde kodu Gemini ile üret; çalıştırmadan önce oku.
3. Kontrol hücresi geçerse bir **tamamlama kodu** verir; onu ML Laboratuvarı'ndaki kutuya yapıştır.

> Not: 2025 Ekim'in değeri eksik (ABD federal hükümetinin kapanması sırasında anket derlenemedi). Doldurma; nasıl ele alındığını denetle.

In [ ]:
#@title Kurulum — çalıştır, değiştirme

import io, base64, urllib.request
import numpy as np
import pandas as pd

def fred_indir(seri_kodu):
    """FRED'den bir seriyi indirir; sütunlar: tarih, <seri_kodu>. FRED eksik değerleri '.' ile gösterir;
    bunlar bilerek NaN yapılır ve sayısı yazdırılır."""
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + seri_kodu
    with urllib.request.urlopen(url) as yanit:
        df = pd.read_csv(io.BytesIO(yanit.read()))
    df.columns = ["tarih", seri_kodu]
    df["tarih"] = pd.to_datetime(df["tarih"])
    df[seri_kodu] = pd.to_numeric(df[seri_kodu], errors="coerce")
    print(f"{seri_kodu}: {len(df)} satır, {df[seri_kodu].isna().sum()} eksik değer, {df['tarih'].min():%Y-%m} – {df['tarih'].max():%Y-%m}")
    return df

issizlik = fred_indir("UNRATE").rename(columns={"UNRATE": "oran"}).sort_values("tarih").reset_index(drop=True)
_kopya = issizlik.copy()
KESIM = pd.Timestamp("2015-01-01")

def _k(s):
    return base64.b64decode(s).decode()

def _beklenen():
    d = _kopya.copy()
    d["onceki"] = d["oran"].shift(1)
    egt = d[d["tarih"] < KESIM].dropna()
    tst = d[d["tarih"] >= KESIM].dropna()
    saf = float(np.sqrt(((tst["oran"] - tst["onceki"]) ** 2).mean()))
    b, a = np.polyfit(egt["onceki"], egt["oran"], 1)
    model = float(np.sqrt(((tst["oran"] - (a + b * tst["onceki"])) ** 2).mean()))
    hepsi = d.dropna()
    b_hepsi, a_hepsi = np.polyfit(hepsi["onceki"], hepsi["oran"], 1)
    return dict(saf=saf, a=a, b=b, model=model, a_hepsi=a_hepsi, b_hepsi=b_hepsi)

def kontrol_A(egitim, test, kok_hko_saf):
    for ad, t in [("egitim", egitim), ("test", test)]:
        if "tarih" not in t.columns or "oran" not in t.columns:
            print(f"❌ '{ad}' tablosunda 'tarih' ve 'oran' sütunları olmalı."); return
    if pd.Timestamp(egitim["tarih"].max()) >= KESIM or pd.Timestamp(test["tarih"].min()) < KESIM:
        print("❌ Bölme zamana göre değil: eğitim 2015 öncesi, test 2015 ve sonrası olmalı. Veri karıştırılmış olabilir."); return
    if pd.Timestamp(egitim["tarih"].min()) > _kopya["tarih"].min() + pd.Timedelta(days=40):
        print("❌ Eğitim verisi serinin başından başlamıyor; bazı yıllar kaybolmuş."); return
    if len(test) < (_kopya["tarih"] >= KESIM).sum() - 2:
        print("❌ Test verisinde beklenenden az ay var."); return
    e = _beklenen()
    if abs(float(kok_hko_saf) - e["saf"]) > 0.02:
        print(f"❌ Saf öngörünün kök HKO'su beklenenden farklı. Öngörü 'geçen ayın oranı' mı? Hata kare alınıp ortalanıp karekökü alındı mı?"); return
    print(f"✅ Tüm kontroller geçti (saf öngörü kök HKO ≈ {e['saf']:.3f}). Tamamlama kodun: " + _k("TUwyLVNBRi04UlRR"))

def kontrol_B(a, b, kok_hko_model):
    e = _beklenen()
    if abs(float(b) - e["b_hepsi"]) < 0.002 and abs(float(b) - e["b"]) > 0.002:
        print("❌ Katsayılar tüm veriyle (test dönemi dahil) kestirilmiş görünüyor: bu bir sızıntı. Yalnızca eğitim verisini kullan."); return
    if abs(float(b) - e["b"]) > 0.005 or abs(float(a) - e["a"]) > 0.03:
        print("❌ Katsayılar beklenenden farklı. Model 'oran_t = a + b · oran_t-1' mi? Yalnızca eğitim verisiyle mi kestirildi?"); return
    if abs(float(kok_hko_model) - e["model"]) > 0.02:
        print("❌ Modelin test kök HKO'su beklenenden farklı. Test döneminde mi hesaplandı?"); return
    print(f"✅ Tüm kontroller geçti (model ≈ {e['model']:.3f}, saf öngörü ≈ {e['saf']:.3f}). Tamamlama kodun: " + _k("TUwyLU1ETC01SFdD"))

print("issizlik tablosu:", issizlik.shape)
issizlik.tail()

## Görev A — Zamana göre bölme ve kıyas ölçütü

1. `issizlik` tablosunu **zamana göre** böl: `tarih < 2015-01-01` → `egitim`, `tarih >= 2015-01-01` → `test`. Karıştırma yok.
2. Test döneminde **saf öngörünün** (bu ay = geçen ay) kök HKO'sunu hesapla ve `kok_hko_saf` adlı değişkene ata.

**Denetim soruları:** `train_test_split` ya da `sample` var mı? Kesim tarihi doğru mu? Eksik ay hatayı bozuyor mu?

In [ ]:
# Görev A: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol A — çalıştır, değiştirme

In [ ]:
kontrol_A(egitim, test, kok_hko_saf)

## Görev B — Basit bir regresyon

`oran_t = a + b · oran_t-1` modelini **yalnızca `egitim` verisiyle** kestir. Katsayıları `a` ve `b` değişkenlerine, test dönemindeki kök HKO'yu `kok_hko_model` değişkenine ata ve saf öngörüyle karşılaştır.

**Denetim soruları:** Katsayılar hangi veriyle kestirildi (`pd.concat` var mı)? Test döneminin ilk ayı için geçen ayın değeri nereden geliyor? Model saf öngörüyü ne kadar geçiyor?

In [ ]:
# Görev B: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol B — çalıştır, değiştirme

In [ ]:
kontrol_B(a, b, kok_hko_model)

## Denetim notların

- Gemini veriyi nasıl böldü? Değiştirmen gerekti mi?
- Regresyon katsayıları hangi veriyle kestirildi?
- Model saf öngörüyü geçiyor mu, ne kadar?